预处理pp md，去掉空文件，翻译为英文，拆分为2k segment=，一个文件输出为一个文件，里面记录该文件拆分结果，以json格式保存即可
遍历指定目录的所有md文件1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch，然后将内容很短，一看就不是privacy policy的content的md给移出去;
剩下的文件先拆分

In [ ]:
# {f_position}\{s_position}
f_position = "AA2_second_batch"
s_position = "AA6_sisth_100_batch" # AA6_sisth_100_batch, AA7_seventh_100_batch # AA3_third_100_batch, AA4_forth_100_batch, AA5_fifth_100_batch

In [52]:
# 如果没装翻译库，先安装一次：
# %pip install deep-translator

import re
import json
import shutil
from pathlib import Path
from datetime import datetime

# try:
#     from deep_translator import GoogleTranslator
#     HAS_TRANSLATOR = True
# except Exception:
#     HAS_TRANSLATOR = False
HAS_TRANSLATOR = False

In [53]:
# ====== 配置 ======
# SRC_DIR1 = Path(r"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch")

# 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA3_third_100_batch\privacy_policy_md



IN_DIR= Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\privacy_policy_md")
# SRC_DIR = Path(r"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA1_first_batch\AA2_second_100_batch\leftover")
# OUT_JSON_DIR = IN_DIR.parent / f"{IN_DIR.name}_seg_json"
# MOVED_EMPTY_DIR = IN_DIR.parent / f"{IN_DIR.name}_moved_empty"
OUT_DIR = Path(rf"1122apk\1122apk_privacy_policy_url\pp_md_2024_10\{f_position}\{s_position}\split_segment")
OUT_JSON_DIR = OUT_DIR
# MOVED_EMPTY_DIR = OUT_DIR.parent / f"{IN_DIR.name}_moved_empty"

SEGMENT_MAX_CHARS = 2000
# MIN_EFFECTIVE_CHARS = 400   # 太短直接判无效 
# MIN_POLICY_KEYWORDS = 2     # 隐私关键词命中阈值 

PRIVACY_KEYWORDS = [
    "privacy", "personal data", "personal information", "data collection",
    "data retention", "cookie", "gdpr", "ccpa", "processor", "controller",
    "third party", "data sharing", "security", "your rights", "contact us"
]

TOPIC_GROUPS = [
    ["collect", "information we collect", "data we collect"],
    ["use", "how we use"],
    ["share", "third party", "disclose"],
    ["security", "protect"],
    ["rights", "access", "delete"],
    ["contact", "email", "address"],
]

OUT_JSON_DIR.mkdir(parents=True, exist_ok=True)
# MOVED_EMPTY_DIR.mkdir(parents=True, exist_ok=True)
# MOVED_NON_POLICY_DIR.mkdir(parents=True, exist_ok=True)

In [54]:
# translator = GoogleTranslator(source="auto", target="en") if HAS_TRANSLATOR else None

def clean_text(md_text: str) -> str:
    text = md_text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()
    return text


def is_probably_english(text: str) -> bool:
    # 简单规则：ASCII 可打印字符比例高，且英文字母占比不低
    if not text:
        return True
    sample = text[:5000]
    ascii_printable = sum(1 for c in sample if 32 <= ord(c) <= 126 or c in "\n\t")
    letters = sum(1 for c in sample if ("a" <= c.lower() <= "z"))
    ratio_ascii = ascii_printable / max(len(sample), 1)
    ratio_letters = letters / max(len(sample), 1)
    return ratio_ascii >= 0.85 and ratio_letters >= 0.15


# def translate_to_english(text: str) -> tuple[str, bool, str]:
#     if not text:
#         return text, False, "empty"
#     if is_probably_english(text):
#         return text, False, "already_english"
#     if not HAS_TRANSLATOR:
#         return text, False, "translator_not_installed"

#     # 分块翻译，避免单次太长失败
#     chunks = []
#     lines = text.split("\n")
#     buf = []
#     cur_len = 0
#     max_chunk = 3500

#     for line in lines:
#         add_len = len(line) + 1
#         if cur_len + add_len > max_chunk and buf:
#             chunks.append("\n".join(buf))
#             buf = [line]
#             cur_len = add_len
#         else:
#             buf.append(line)
#             cur_len += add_len
#     if buf:
#         chunks.append("\n".join(buf))

#     out = []
#     try:
#         for ch in chunks:
#             out.append(translator.translate(ch))
#         return "\n".join(out), True, "translated"
#     except Exception as e:
#         return text, False, f"translate_failed: {e}"

In [55]:
# def looks_like_policy_(text: str) -> bool:
#     effective = re.sub(r"\s+", " ", text).strip()
#     if len(effective) < MIN_EFFECTIVE_CHARS:
#         return False

#     low = effective.lower()
#     hits = sum(1 for k in PRIVACY_KEYWORDS if k in low)

#     return hits >= MIN_POLICY_KEYWORDS

# def looks_like_policy(text: str) -> bool:
#     effective = re.sub(r"\s+", " ", text).strip()

#     # 去掉 URL 再算长度，避免被一条链接“抬高”
#     effective_no_url = re.sub(r"https?://\S+", "", effective).strip()
#     if len(effective_no_url) < MIN_EFFECTIVE_CHARS:
#         return False

#     low = effective.lower()
#     hits = sum(1 for k in PRIVACY_KEYWORDS if k in low)

#     group_hits = sum(any(k in low for k in g) for g in TOPIC_GROUPS)
#     return hits >= MIN_POLICY_KEYWORDS and group_hits >= 2


def split_into_segments(text: str, max_chars: int = 2000) -> list[str]:
    # 按段落优先切分，不够再按句子/硬切
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    segments = []
    cur = ""

    for p in paras:
        if not cur:
            if len(p) <= max_chars:
                cur = p
            else:
                # 段落太长，按句号切
                sentences = re.split(r"(?<=[.!?])\s+", p)
                tmp = ""
                for s in sentences:
                    if len(tmp) + len(s) + 1 <= max_chars:
                        tmp = (tmp + " " + s).strip()
                    else:
                        if tmp:
                            segments.append(tmp)
                        if len(s) <= max_chars:
                            tmp = s
                        else:
                            # 再硬切
                            for i in range(0, len(s), max_chars):
                                segments.append(s[i:i+max_chars])
                            tmp = ""
                if tmp:
                    cur = tmp
        else:
            if len(cur) + 2 + len(p) <= max_chars:
                cur = cur + "\n\n" + p
            else:
                segments.append(cur)
                if len(p) <= max_chars:
                    cur = p
                else:
                    # 超长段硬切
                    for i in range(0, len(p), max_chars):
                        part = p[i:i+max_chars]
                        if len(part) == max_chars:
                            segments.append(part)
                        else:
                            cur = part

    if cur:
        segments.append(cur)

    return [s.strip() for s in segments if s.strip()]

def get_all_files(directory):
    p = Path(directory)
    # 只取一级文件；只要 .md；排序保证稳定
    files = [
        str(x) for x in p.iterdir()
        if x.is_file() and x.suffix.lower() == ".md"
    ]
    return sorted(files)

In [56]:
summary = {
    "total_md": 0,
    "moved_empty": 0,
    "processed_json": 0,
    "translate_success": 0,
    "translate_skipped_or_failed": 0
}
all_files = get_all_files(IN_DIR) #(SRC_DIR)
for md_file in all_files:
    summary["total_md"] += 1
    md_file = Path(md_file)
    raw = md_file.read_text(encoding="utf-8", errors="ignore")
    text = clean_text(raw)

    # if not text or not looks_like_policy(text):
    #     shutil.move(str(md_file), str(MOVED_EMPTY_DIR / md_file.name))
    #     summary["moved_empty"] += 1
    #     continue

    

    # if not looks_like_policy(text):
    #     shutil.move(str(md_file), str(MOVED_NON_POLICY_DIR / md_file.name))
    #     summary["moved_nonpolicy"] += 1
    #     continue

    # en_text, translated, translate_note = translate_to_english(text)
    # if translated:
    #     summary["translate_success"] += 1
    # else:
    #     summary["translate_skipped_or_failed"] += 1

    segments = split_into_segments(text, max_chars=SEGMENT_MAX_CHARS)

    out_obj = {
        "source_file": md_file.name,
        "source_path": str(md_file),
        "processed_at": datetime.now().isoformat(timespec="seconds"),
        # "translated_to_english": translated,
        # "translate_note": translate_note,
        "segment_max_chars": SEGMENT_MAX_CHARS,
        "segment_count": len(segments),
        "segments": [
            {
                "segment_id": i + 1,
                "char_len": len(seg),
                "text": seg
            }
            for i, seg in enumerate(segments)
        ]
    }

    out_json = OUT_JSON_DIR / f"{md_file.stem}.json"
    out_json.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2), encoding="utf-8")
    summary["processed_json"] += 1

print("Done.")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("OUT_JSON_DIR:", OUT_JSON_DIR)
# print("MOVED_EMPTY_DIR:", MOVED_EMPTY_DIR)
# print("MOVED_NON_POLICY_DIR:", MOVED_NON_POLICY_DIR)

Done.
{
  "total_md": 20,
  "moved_empty": 0,
  "processed_json": 20,
  "translate_success": 0,
  "translate_skipped_or_failed": 0
}
OUT_JSON_DIR: 1122apk\1122apk_privacy_policy_url\pp_md_2024_10\AA2_second_batch\AA5_fifth_100_batch\split_segment
